In [1]:
# ============================================================
# CELL 1 — INSTALL, MOUNT DRIVE, LOCATE AND VALIDATE FILES
# ============================================================

%pip install -q -U \
    "transformers>=4.48,<5" \
    "sentence-transformers>=3.4,<6" \
    accelerate \
    bitsandbytes \
    faiss-cpu \
    rank-bm25 \
    rapidfuzz \
    sacrebleu \
    nltk \
    bert-score==0.3.13 \
    pandas \
    tqdm

# These packages are not needed here and sometimes conflict
# with Transformers in Colab.
%pip uninstall -y -q torchvision timm

from google.colab import drive, files

drive.mount("/content/drive", force_remount=False)

import gc
import hashlib
import json
import os
import pickle
import random
import re
import unicodedata

from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch

from IPython.display import display
from rank_bm25 import BM25Okapi
from rapidfuzz import fuzz
from tqdm.auto import tqdm


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# Artifact directory
# ------------------------------------------------------------

ART_DIR = Path(
    "/content/drive/MyDrive/govt_rag_artifacts/RAG_3"
)

ART_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Expected input files
# ------------------------------------------------------------

SEARCH_DIRS = [
    Path("/content"),
    Path("/content/drive/MyDrive/govt_rag_artifacts"),
    ART_DIR,
]

KB_PATTERNS = {
    "passport": [
        "passport_clean_kb_updated*.json"
    ],

    "nid": [
        "nid_clean_kb_updated*.json"
    ],

    "birth_death": [
        "birth_death_clean_kb_updated*.json"
    ],

    "tin": [
        "TIN_clean_kb_updated*.json",
        "tin_clean_kb_updated*.json"
    ],
}

TEST_PATTERNS = {
    "passport": [
        "passport_qa_test*.json"
    ],

    "nid": [
        "nid_qa_test*.json"
    ],

    "birth_death": [
        "birth_death_qa_test*.json"
    ],

    "tin": [
        "TIN_qa_test*.json",
        "tin_qa_test*.json"
    ],
}


# These counts ensure that the old KB files are not selected
# accidentally.

EXPECTED_KB_COUNTS = {
    "passport": 84,
    "nid": 398,
    "birth_death": 93,
    "tin": 102,
}

EXPECTED_TEST_COUNTS = {
    "passport": 63,
    "nid": 74,
    "birth_death": 63,
    "tin": 48,
}


# ------------------------------------------------------------
# Utility functions
# ------------------------------------------------------------

def load_json(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as file:

        return json.load(file)


def resolve_patterns(patterns):

    matches = []

    for directory in SEARCH_DIRS:

        if not directory.exists():
            continue

        for pattern in patterns:
            matches.extend(
                directory.glob(pattern)
            )

    matches = [
        path
        for path in matches
        if path.is_file()
    ]

    if not matches:
        return None

    # If Colab renamed a duplicate file with (1),
    # choose the most recently uploaded version.
    return max(
        matches,
        key=lambda path: path.stat().st_mtime
    )


def resolve_all():

    kb_files = {
        domain: resolve_patterns(patterns)
        for domain, patterns
        in KB_PATTERNS.items()
    }

    test_files = {
        domain: resolve_patterns(patterns)
        for domain, patterns
        in TEST_PATTERNS.items()
    }

    return kb_files, test_files


# ------------------------------------------------------------
# Find files or request upload
# ------------------------------------------------------------

KB_FILES, TEST_FILES = resolve_all()

missing = [

    f"KB: {domain}"

    for domain, path in KB_FILES.items()

    if path is None

] + [

    f"Test: {domain}"

    for domain, path in TEST_FILES.items()

    if path is None
]


if missing:

    print(
        "Upload the 4 updated KB files and "
        "the 4 test QA files now."
    )

    files.upload()

    KB_FILES, TEST_FILES = resolve_all()


still_missing = [

    f"KB: {domain}"

    for domain, path in KB_FILES.items()

    if path is None

] + [

    f"Test: {domain}"

    for domain, path in TEST_FILES.items()

    if path is None
]


if still_missing:

    raise FileNotFoundError(
        "Missing files: "
        + ", ".join(still_missing)
    )


# ------------------------------------------------------------
# Validate updated KB files
# ------------------------------------------------------------

for domain, path in KB_FILES.items():

    rows = load_json(path)

    if not isinstance(rows, list):

        raise ValueError(
            f"{path.name} must contain a JSON list."
        )

    if len(rows) != EXPECTED_KB_COUNTS[domain]:

        raise ValueError(
            f"{domain} updated KB should have "
            f"{EXPECTED_KB_COUNTS[domain]} records, "
            f"but {len(rows)} were found.\n"
            f"Wrong file selected: {path}"
        )

    if any(
        not str(row.get("text", "")).strip()
        for row in rows
    ):

        raise ValueError(
            f"{path.name} contains a blank text field."
        )


# ------------------------------------------------------------
# Validate test files
# ------------------------------------------------------------

for domain, path in TEST_FILES.items():

    rows = load_json(path)

    if not isinstance(rows, list):

        raise ValueError(
            f"{path.name} must contain a JSON list."
        )

    if len(rows) != EXPECTED_TEST_COUNTS[domain]:

        raise ValueError(
            f"{domain} test file should have "
            f"{EXPECTED_TEST_COUNTS[domain]} records, "
            f"but {len(rows)} were found.\n"
            f"Wrong file selected: {path}"
        )

    for row in rows:

        if not str(
            row.get("instruction", "")
        ).strip():

            raise ValueError(
                f"{path.name} contains "
                "a blank instruction."
            )

        if not str(
            row.get("output", "")
        ).strip():

            raise ValueError(
                f"{path.name} contains "
                "a blank output/gold answer."
            )


# ------------------------------------------------------------
# Save input information
# ------------------------------------------------------------

input_manifest = {

    "kb_files": {
        key: str(value)
        for key, value in KB_FILES.items()
    },

    "test_files": {
        key: str(value)
        for key, value in TEST_FILES.items()
    },

    "kb_counts": EXPECTED_KB_COUNTS,
    "test_counts": EXPECTED_TEST_COUNTS,
}


with open(
    ART_DIR / "RAG_3_input_files.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        input_manifest,
        file,
        ensure_ascii=False,
        indent=2
    )


print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "Not available"
)

print("Artifact directory:", ART_DIR)

print("\nUpdated KB files:")

for domain, path in KB_FILES.items():

    print(
        f"  {domain:12s} "
        f"{EXPECTED_KB_COUNTS[domain]:3d}  "
        f"{path.name}"
    )


print("\nTest files:")

for domain, path in TEST_FILES.items():

    print(
        f"  {domain:12s} "
        f"{EXPECTED_TEST_COUNTS[domain]:3d}  "
        f"{path.name}"
    )

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 127.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80

In [2]:
# ============================================================
# CELL 2 — BUILD COMPLETE UPDATED CORPUS AND TEST TABLE
# ============================================================


def normalize_domain(domain):

    value = str(domain).strip().lower()

    if "passport" in value:
        return "passport"

    if "birth" in value or "death" in value:
        return "birth_death"

    if "nid" in value:
        return "nid"

    if "tin" in value or "tax" in value:
        return "tin"

    return value


def split_words(
    text,
    max_words=220,
    overlap=35
):

    words = str(text).split()

    if len(words) <= max_words:
        return [str(text).strip()]

    pieces = []
    start = 0

    while start < len(words):

        end = min(
            start + max_words,
            len(words)
        )

        piece = " ".join(
            words[start:end]
        ).strip()

        if piece:
            pieces.append(piece)

        if end >= len(words):
            break

        start = max(
            0,
            end - overlap
        )

    return pieces


def build_index_text(item, text):

    parts = []

    title = str(
        item.get("title", "")
    ).strip()

    topic = str(
        item.get("topic", "")
    ).strip()

    service = str(
        item.get("service", "")
    ).strip()

    source_name = str(
        item.get("source_name", "")
    ).strip()

    keywords = item.get(
        "keywords",
        []
    )

    if isinstance(keywords, list):

        keywords = ", ".join(
            str(value)
            for value in keywords
            if str(value).strip()
        )

    else:

        keywords = str(
            keywords
        ).strip()


    if title:
        parts.append(
            f"শিরোনাম: {title}"
        )

    if topic:
        parts.append(
            f"বিষয়: {topic}"
        )

    if service:
        parts.append(
            f"সেবা: {service}"
        )

    if keywords:
        parts.append(
            f"কীওয়ার্ড: {keywords}"
        )

    if source_name:
        parts.append(
            f"উৎস: {source_name}"
        )

    parts.append(
        f"তথ্য: {text}"
    )

    return "\n".join(parts)


# ------------------------------------------------------------
# Build chunks from all four complete updated KBs
# ------------------------------------------------------------

all_chunks = []
record_counts = {}


for domain_key, path in KB_FILES.items():

    records = load_json(path)

    record_counts[domain_key] = len(records)


    for record_number, item in enumerate(
        records,
        start=1
    ):

        text = str(
            item.get("text", "")
        ).strip()

        domain = normalize_domain(
            item.get(
                "domain",
                domain_key
            )
        )

        doc_id = str(
            item.get("doc_id", "")
        ).strip()

        if not doc_id:

            doc_id = (
                f"{domain}_"
                f"{record_number:04d}"
            )


        pieces = split_words(text)


        for part_number, piece in enumerate(
            pieces,
            start=1
        ):

            chunk_position = len(
                all_chunks
            )

            all_chunks.append({

                "faiss_id":
                    chunk_position,

                "chunk_id":
                    f"rag3_chunk_"
                    f"{chunk_position:05d}",

                "doc_id":
                    doc_id,

                "part_id":
                    part_number,

                "domain":
                    domain,

                "service":
                    str(
                        item.get(
                            "service",
                            ""
                        )
                    ).strip(),

                "topic":
                    str(
                        item.get(
                            "topic",
                            ""
                        )
                    ).strip(),

                "title":
                    str(
                        item.get(
                            "title",
                            ""
                        )
                    ).strip(),

                "source_name":
                    str(
                        item.get(
                            "source_name",
                            ""
                        )
                    ).strip(),

                "source_url":
                    str(
                        item.get(
                            "source_url",
                            ""
                        )
                    ).strip(),

                "last_checked":
                    str(
                        item.get(
                            "last_checked",
                            ""
                        )
                    ).strip(),

                "text":
                    piece,

                "index_text":
                    build_index_text(
                        item,
                        piece
                    ),
            })


if not all_chunks:

    raise ValueError(
        "No KB chunks were created."
    )


for expected_id, chunk in enumerate(
    all_chunks
):

    if chunk["faiss_id"] != expected_id:

        raise RuntimeError(
            "FAISS IDs are not sequential."
        )


# ------------------------------------------------------------
# Save merged updated corpus
# ------------------------------------------------------------

chunks_json_path = (
    ART_DIR
    / "RAG_3_merged_kb_chunks.json"
)

chunks_csv_path = (
    ART_DIR
    / "RAG_3_merged_kb_chunks.csv"
)


with open(
    chunks_json_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        all_chunks,
        file,
        ensure_ascii=False,
        indent=2
    )


pd.DataFrame(
    all_chunks
).to_csv(

    chunks_csv_path,

    index=False,

    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# Calculate exact corpus hash
# ------------------------------------------------------------

canonical_corpus = json.dumps(

    all_chunks,

    ensure_ascii=False,

    sort_keys=True,

    separators=(",", ":")
)


corpus_sha256 = hashlib.sha256(

    canonical_corpus.encode(
        "utf-8"
    )

).hexdigest()


# ------------------------------------------------------------
# Build combined test table
# ------------------------------------------------------------

test_rows = []


for domain_key, path in TEST_FILES.items():

    for item in load_json(path):

        test_rows.append({

            "domain":
                normalize_domain(
                    item.get(
                        "domain",
                        domain_key
                    )
                ),

            "id":
                str(
                    item.get(
                        "id",
                        ""
                    )
                ).strip(),

            "question":
                str(
                    item.get(
                        "instruction",
                        ""
                    )
                ).strip(),

            "gold_answer":
                str(
                    item.get(
                        "output",
                        ""
                    )
                ).strip(),

            "source_url":
                str(
                    item.get(
                        "source_url",
                        ""
                    )
                ).strip(),

            "topic":
                str(
                    item.get(
                        "topic",
                        ""
                    )
                ).strip(),

            "question_type":
                str(
                    item.get(
                        "question_type",
                        ""
                    )
                ).strip(),
        })


expected_tests = sum(
    EXPECTED_TEST_COUNTS.values()
)


if len(test_rows) != expected_tests:

    raise ValueError(
        f"Expected {expected_tests} tests, "
        f"but created {len(test_rows)}."
    )


if any(
    not row["gold_answer"]
    for row in test_rows
):

    raise ValueError(
        "At least one test row "
        "has a blank gold answer."
    )


tests_json_path = (
    ART_DIR
    / "RAG_3_test_rows.json"
)

tests_csv_path = (
    ART_DIR
    / "RAG_3_test_rows.csv"
)


with open(
    tests_json_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        test_rows,
        file,
        ensure_ascii=False,
        indent=2
    )


pd.DataFrame(
    test_rows
).to_csv(

    tests_csv_path,

    index=False,

    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# Save corpus information
# ------------------------------------------------------------

chunks_by_domain = {

    str(domain): int(count)

    for domain, count in (

        pd.DataFrame(
            all_chunks
        )
        .groupby("domain")
        .size()
        .items()
    )
}


corpus_info = {

    "corpus_sha256":
        corpus_sha256,

    "total_kb_records":
        sum(
            record_counts.values()
        ),

    "total_chunks":
        len(all_chunks),

    "total_tests":
        len(test_rows),

    "records_by_domain":
        record_counts,

    "chunks_by_domain":
        chunks_by_domain,
}


with open(
    ART_DIR / "RAG_3_corpus_info.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        corpus_info,
        file,
        ensure_ascii=False,
        indent=2
    )


print(
    "Total updated KB records:",
    corpus_info["total_kb_records"]
)

print(
    "Total indexed chunks:",
    corpus_info["total_chunks"]
)

print(
    "Total test questions:",
    corpus_info["total_tests"]
)

print(
    "Corpus SHA256:",
    corpus_sha256
)


display(

    pd.DataFrame(
        all_chunks
    )
    .groupby("domain")
    .size()
    .rename("chunks")
    .reset_index()
)

Total updated KB records: 677
Total indexed chunks: 677
Total test questions: 248
Corpus SHA256: b877614ead9315e9af645c5f42cf624cc69ab6107622c2f249cebcffa4b29a60


,domain,chunks
0,birth_death,93
1,nid,398
2,passport,84
3,tin,102


In [3]:
# ============================================================
# CELL 3 — BUILD AND STORE BGE-M3 + BM25 INDICES
# ============================================================

from sentence_transformers import SentenceTransformer


EMBED_MODEL_NAME = "BAAI/bge-m3"

# False means:
# - Build when no matching index exists.
# - Reuse when the stored index matches this exact KB.
FORCE_REBUILD = False


CHUNKS_PATH = (
    ART_DIR
    / "RAG_3_merged_kb_chunks.json"
)

CORPUS_INFO_PATH = (
    ART_DIR
    / "RAG_3_corpus_info.json"
)

EMBEDDINGS_PATH = (
    ART_DIR
    / "RAG_3_bge_m3_embeddings.npy"
)

FAISS_PATH = (
    ART_DIR
    / "RAG_3_bge_m3_faiss.index"
)

BM25_PATH = (
    ART_DIR
    / "RAG_3_bm25_index.pkl"
)

MANIFEST_PATH = (
    ART_DIR
    / "RAG_3_index_manifest.json"
)


chunks = load_json(
    CHUNKS_PATH
)

corpus_info = load_json(
    CORPUS_INFO_PATH
)

index_texts = [
    chunk["index_text"]
    for chunk in chunks
]


BANGLA_DIGITS = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)


def bm25_tokenize(text):

    value = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    value = value.translate(
        BANGLA_DIGITS
    ).lower()

    return re.findall(
        r"[\u0980-\u09FF]+|"
        r"[a-z]+|"
        r"\d+(?:\.\d+)?",
        value
    )


def existing_index_is_valid():

    required = [
        EMBEDDINGS_PATH,
        FAISS_PATH,
        BM25_PATH,
        MANIFEST_PATH,
    ]

    if not all(
        path.exists()
        for path in required
    ):

        return False


    try:

        manifest = load_json(
            MANIFEST_PATH
        )

        return (

            manifest.get(
                "corpus_sha256"
            )
            ==
            corpus_info[
                "corpus_sha256"
            ]

            and

            manifest.get(
                "embedding_model"
            )
            ==
            EMBED_MODEL_NAME

            and

            int(
                manifest.get(
                    "total_chunks",
                    -1
                )
            )
            ==
            len(chunks)
        )

    except Exception:

        return False


reuse = (

    existing_index_is_valid()

    and

    not FORCE_REBUILD
)


if reuse:

    print(
        "Matching RAG-3 indices already exist. "
        "Reusing them."
    )


else:

    device = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    batch_size = (
        8
        if device == "cuda"
        else 2
    )


    print(
        "Loading BGE-M3 on:",
        device
    )


    embedder = SentenceTransformer(
        EMBED_MODEL_NAME,
        device=device
    )

    # KB records are short.
    embedder.max_seq_length = 1024


    print(
        "Encoding the complete updated KB..."
    )


    embeddings = embedder.encode(

        index_texts,

        batch_size=batch_size,

        show_progress_bar=True,

        convert_to_numpy=True,

        normalize_embeddings=True

    ).astype("float32")


    if (

        embeddings.ndim != 2

        or

        len(embeddings) != len(chunks)

    ):

        raise RuntimeError(
            f"Embedding shape is invalid: "
            f"{embeddings.shape}; expected "
            f"{len(chunks)} rows."
        )


    # --------------------------------------------------------
    # Store dense embeddings
    # --------------------------------------------------------

    np.save(
        EMBEDDINGS_PATH,
        embeddings
    )


    # --------------------------------------------------------
    # Build exact FAISS cosine index
    # --------------------------------------------------------

    dense_index = faiss.IndexFlatIP(
        embeddings.shape[1]
    )

    dense_index.add(
        embeddings
    )

    faiss.write_index(
        dense_index,
        str(FAISS_PATH)
    )


    # --------------------------------------------------------
    # Build and store BM25
    # --------------------------------------------------------

    print(
        "Building BM25 index..."
    )


    tokenized_corpus = [

        bm25_tokenize(text)

        for text in index_texts
    ]


    if any(
        len(tokens) == 0
        for tokens in tokenized_corpus
    ):

        empty_ids = [

            index

            for index, tokens
            in enumerate(
                tokenized_corpus
            )

            if not tokens
        ]

        raise ValueError(
            "BM25 found empty documents at IDs: "
            f"{empty_ids[:10]}"
        )


    bm25 = BM25Okapi(
        tokenized_corpus
    )


    with open(
        BM25_PATH,
        "wb"
    ) as file:

        pickle.dump(

            {
                "bm25":
                    bm25,

                "tokenized_corpus":
                    tokenized_corpus,

                "tokenizer":
                    "NFKC + Bangla digits to English "
                    "+ Bangla/English/number regex",
            },

            file,

            protocol=pickle.HIGHEST_PROTOCOL
        )


    # --------------------------------------------------------
    # Store index manifest
    # --------------------------------------------------------

    manifest = {

        "created_at_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "corpus_sha256":
            corpus_info[
                "corpus_sha256"
            ],

        "embedding_model":
            EMBED_MODEL_NAME,

        "embedding_dimension":
            int(
                embeddings.shape[1]
            ),

        "normalized_embeddings":
            True,

        "faiss_index_type":
            "IndexFlatIP",

        "bm25_type":
            "BM25Okapi",

        "total_kb_records":
            int(
                corpus_info[
                    "total_kb_records"
                ]
            ),

        "total_chunks":
            len(chunks),

        "artifacts": {

            "chunks":
                CHUNKS_PATH.name,

            "embeddings":
                EMBEDDINGS_PATH.name,

            "faiss":
                FAISS_PATH.name,

            "bm25":
                BM25_PATH.name,
        },
    }


    with open(
        MANIFEST_PATH,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            manifest,
            file,
            ensure_ascii=False,
            indent=2
        )


    del (
        embedder,
        embeddings,
        dense_index,
        bm25,
        tokenized_corpus
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ------------------------------------------------------------
# Verify all stored index files
# ------------------------------------------------------------

manifest = load_json(
    MANIFEST_PATH
)

dense_index = faiss.read_index(
    str(FAISS_PATH)
)

embeddings_shape = np.load(
    EMBEDDINGS_PATH,
    mmap_mode="r"
).shape


with open(
    BM25_PATH,
    "rb"
) as file:

    bm25_bundle = pickle.load(
        file
    )


if dense_index.ntotal != len(chunks):

    raise RuntimeError(
        f"FAISS contains "
        f"{dense_index.ntotal} vectors; "
        f"expected {len(chunks)}."
    )


if embeddings_shape[0] != len(chunks):

    raise RuntimeError(
        f"Embedding file contains "
        f"{embeddings_shape[0]} rows; "
        f"expected {len(chunks)}."
    )


if (
    bm25_bundle["bm25"].corpus_size
    !=
    len(chunks)
):

    raise RuntimeError(
        f"BM25 contains "
        f"{bm25_bundle['bm25'].corpus_size} "
        f"documents; expected {len(chunks)}."
    )


print(
    "\nIndex verification successful."
)

print(
    "FAISS vectors:",
    dense_index.ntotal
)

print(
    "Embedding shape:",
    embeddings_shape
)

print(
    "BM25 documents:",
    bm25_bundle["bm25"].corpus_size
)

print(
    "Saved in:",
    ART_DIR
)

print("\nManifest:")

display(
    pd.DataFrame([manifest])
)

Loading BGE-M3 on: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Encoding the complete updated KB...


Batches:   0%|          | 0/85 [00:00<?, ?it/s]

Building BM25 index...

Index verification successful.
FAISS vectors: 677
Embedding shape: (677, 1024)
BM25 documents: 677
Saved in: /content/drive/MyDrive/govt_rag_artifacts/RAG_3

Manifest:


,created_at_utc,corpus_sha256,embedding_model,embedding_dimension,normalized_embeddings,faiss_index_type,bm25_type,total_kb_records,total_chunks,artifacts
0,2026-08-01T17:41:10.019579+00:00,b877614ead9315e9af645c5f42cf624cc69ab6107622c2...,BAAI/bge-m3,1024,True,IndexFlatIP,BM25Okapi,677,677,"{'chunks': 'RAG_3_merged_kb_chunks.json', 'emb..."


In [7]:
# ============================================================
# CELL 4 — HYBRID RETRIEVAL, RERANKING AND QWEN RAG
# ============================================================

from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from sentence_transformers import SentenceTransformer


if not torch.cuda.is_available():

    raise RuntimeError(
        "GPU is required for Cell 4. "
        "In Colab select Runtime > "
        "Change runtime type > T4 GPU."
    )


# ------------------------------------------------------------
# Paths and model settings
# ------------------------------------------------------------

CHUNKS_PATH = (
    ART_DIR
    / "RAG_3_merged_kb_chunks.json"
)

TESTS_PATH = (
    ART_DIR
    / "RAG_3_test_rows.json"
)

FAISS_PATH = (
    ART_DIR
    / "RAG_3_bge_m3_faiss.index"
)

BM25_PATH = (
    ART_DIR
    / "RAG_3_bm25_index.pkl"
)

MANIFEST_PATH = (
    ART_DIR
    / "RAG_3_index_manifest.json"
)

RETRIEVAL_CACHE_PATH = (
    ART_DIR
    / "RAG_3_retrieval_cache.json"
)

PARTIAL_PATH = (
    ART_DIR
    / "RAG_3_predictions_partial.jsonl"
)

PREDICTIONS_CSV_PATH = (
    ART_DIR
    / "RAG_3_predictions.csv"
)

PREDICTIONS_JSON_PATH = (
    ART_DIR
    / "RAG_3_predictions.json"
)


EMBED_MODEL_NAME = (
    "BAAI/bge-m3"
)

RERANKER_NAME = (
    "BAAI/bge-reranker-v2-m3"
)

GENERATOR_NAME = (
    "Qwen/Qwen2.5-7B-Instruct"
)


DENSE_K = 100
BM25_K = 100

RERANK_CANDIDATES = 25
FINAL_K = 5

RRF_CONSTANT = 60

USE_RERANKER = True

MAX_NEW_TOKENS = 400


# ------------------------------------------------------------
# Load stored corpus and indices
# ------------------------------------------------------------

chunks = load_json(
    CHUNKS_PATH
)

test_rows = load_json(
    TESTS_PATH
)

manifest = load_json(
    MANIFEST_PATH
)

dense_index = faiss.read_index(
    str(FAISS_PATH)
)


with open(
    BM25_PATH,
    "rb"
) as file:

    bm25_bundle = pickle.load(
        file
    )

    bm25 = bm25_bundle["bm25"]


if dense_index.ntotal != len(chunks):

    raise RuntimeError(
        "FAISS/chunk count mismatch. "
        "Run Cell 3 again."
    )


BANGLA_DIGITS = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)


def bm25_tokenize(text):

    value = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    value = value.translate(
        BANGLA_DIGITS
    ).lower()

    return re.findall(
        r"[\u0980-\u09FF]+|"
        r"[a-z]+|"
        r"\d+(?:\.\d+)?",
        value
    )


def normalize_url(url):

    value = str(
        url
    ).strip().lower()

    value = re.sub(
        r"^https?://",
        "",
        value
    )

    value = re.sub(
        r"^www\.",
        "",
        value
    )

    return value.rstrip("/")


def token_recall(
    reference,
    context
):

    reference_tokens = bm25_tokenize(
        reference
    )

    context_tokens = bm25_tokenize(
        context
    )

    if not reference_tokens:
        return 0.0

    reference_counts = Counter(
        reference_tokens
    )

    context_counts = Counter(
        context_tokens
    )

    matched = sum(
        (
            reference_counts
            &
            context_counts
        ).values()
    )

    return (
        matched
        /
        len(reference_tokens)
    )


# ------------------------------------------------------------
# Load retrieval models
# ------------------------------------------------------------

print(
    "Loading BGE-M3 query encoder..."
)


embedder = SentenceTransformer(
    EMBED_MODEL_NAME,
    device="cuda"
)

embedder.max_seq_length = 1024


reranker_tokenizer = None
reranker_model = None
reranker_enabled = False


if USE_RERANKER:

    try:

        print(
            "Loading multilingual "
            "BGE reranker..."
        )

        reranker_tokenizer = (
            AutoTokenizer
            .from_pretrained(
                RERANKER_NAME
            )
        )

        reranker_model = (
            AutoModelForSequenceClassification
            .from_pretrained(

                RERANKER_NAME,

                torch_dtype=torch.float16,

                low_cpu_mem_usage=True
            )
            .to("cuda")
        )

        reranker_model.eval()

        reranker_enabled = True


    except Exception as error:

        print(
            "Reranker could not be loaded; "
            "continuing with BGE-M3 + "
            "BM25 RRF."
        )

        print(
            type(error).__name__,
            str(error)[:300]
        )

        reranker_tokenizer = None
        reranker_model = None
        reranker_enabled = False

        gc.collect()
        torch.cuda.empty_cache()


# ------------------------------------------------------------
# Reranking function
# ------------------------------------------------------------

def rerank_candidates(
    question,
    candidates,
    batch_size=4
):

    if (
        not reranker_enabled
        or
        not candidates
    ):

        for candidate in candidates:
            candidate[
                "reranker_score"
            ] = None

        return candidates


    scores = []


    for start in range(
        0,
        len(candidates),
        batch_size
    ):

        batch = candidates[
            start:start + batch_size
        ]

        documents = [
            item["index_text"]
            for item in batch
        ]


        encoded = reranker_tokenizer(

            [question] * len(documents),

            documents,

            padding=True,

            truncation=True,

            max_length=512,

            return_tensors="pt"
        )


        encoded = {

            key: value.to("cuda")

            for key, value
            in encoded.items()
        }


        with torch.inference_mode():

            logits = reranker_model(
                **encoded
            ).logits.squeeze(-1)


        scores.extend(
            logits
            .float()
            .cpu()
            .tolist()
        )


    for candidate, score in zip(
        candidates,
        scores
    ):

        candidate[
            "reranker_score"
        ] = float(score)


    return sorted(

        candidates,

        key=lambda item:
            item["reranker_score"],

        reverse=True
    )


# ------------------------------------------------------------
# BGE-M3 + BM25 hybrid retrieval
# ------------------------------------------------------------

def hybrid_retrieve(
    question,
    domain,
    final_k=FINAL_K
):

    domain = normalize_domain(
        domain
    )


    # Dense retrieval
    query_embedding = embedder.encode(

        [question],

        convert_to_numpy=True,

        normalize_embeddings=True,

        show_progress_bar=False

    ).astype("float32")


    dense_scores, dense_ids = (
        dense_index.search(

            query_embedding,

            dense_index.ntotal
        )
    )


    dense_ranked = []


    for score, index_id in zip(
        dense_scores[0],
        dense_ids[0]
    ):

        index_id = int(
            index_id
        )

        if index_id < 0:
            continue

        if (
            chunks[index_id]["domain"]
            !=
            domain
        ):
            continue

        dense_ranked.append(
            (
                index_id,
                float(score)
            )
        )

        if len(dense_ranked) >= DENSE_K:
            break


    # BM25 retrieval
    sparse_scores = bm25.get_scores(
        bm25_tokenize(question)
    )

    sparse_ids = np.argsort(
        sparse_scores
    )[::-1]


    bm25_ranked = []


    for index_id in sparse_ids:

        index_id = int(
            index_id
        )

        score = float(
            sparse_scores[index_id]
        )

        if score <= 0:
            break

        if (
            chunks[index_id]["domain"]
            !=
            domain
        ):
            continue

        bm25_ranked.append(
            (
                index_id,
                score
            )
        )

        if len(bm25_ranked) >= BM25_K:
            break


    # Reciprocal Rank Fusion
    combined = defaultdict(
        lambda: {

            "rrf_score": 0.0,

            "dense_score": None,

            "bm25_score": None,
        }
    )


    for rank, (
        index_id,
        score
    ) in enumerate(
        dense_ranked,
        start=1
    ):

        combined[
            index_id
        ][
            "rrf_score"
        ] += (
            1.0
            /
            (
                RRF_CONSTANT
                +
                rank
            )
        )

        combined[
            index_id
        ][
            "dense_score"
        ] = score


    for rank, (
        index_id,
        score
    ) in enumerate(
        bm25_ranked,
        start=1
    ):

        combined[
            index_id
        ][
            "rrf_score"
        ] += (
            1.0
            /
            (
                RRF_CONSTANT
                +
                rank
            )
        )

        combined[
            index_id
        ][
            "bm25_score"
        ] = score


    if not combined:
        return []


    ranked_ids = sorted(

        combined,

        key=lambda index_id:
            combined[index_id][
                "rrf_score"
            ],

        reverse=True

    )[:RERANK_CANDIDATES]


    candidates = []


    for index_id in ranked_ids:

        item = dict(
            chunks[index_id]
        )

        item.update(
            combined[index_id]
        )

        candidates.append(item)


    candidates = rerank_candidates(
        question,
        candidates
    )


    if not reranker_enabled:

        candidates = sorted(

            candidates,

            key=lambda item:
                item["rrf_score"],

            reverse=True
        )


    return candidates[:final_k]


# ------------------------------------------------------------
# Retrieve contexts for all test questions
# ------------------------------------------------------------

print(
    "\nRetrieving contexts for "
    "all test questions..."
)


retrieval_rows = []


for row in tqdm(
    test_rows,
    desc="Hybrid retrieval"
):

    retrieved = hybrid_retrieve(

        row["question"],

        row["domain"],

        final_k=FINAL_K
    )


    retrieved_text = "\n".join(
        item["text"]
        for item in retrieved
    )


    retrieved_urls = [

        normalize_url(
            item.get(
                "source_url",
                ""
            )
        )

        for item in retrieved
    ]


    test_url = normalize_url(
        row.get(
            "source_url",
            ""
        )
    )


    source_hit = (

        bool(test_url)

        and

        test_url in retrieved_urls
    )


    support_recall = token_recall(

        row["gold_answer"],

        retrieved_text
    )


    retrieval_rows.append({

        **row,

        "context_hit":
            source_hit,

        "context_support_token_recall":
            float(
                support_recall
            ),

        "empty_context":
            len(retrieved) == 0,

        "retrieved":
            retrieved,
    })


with open(
    RETRIEVAL_CACHE_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        retrieval_rows,
        file,
        ensure_ascii=False,
        indent=2
    )


retrieval_audit = pd.DataFrame([

    {
        "domain":
            row["domain"],

        "id":
            row["id"],

        "question":
            row["question"],

        "context_hit":
            row["context_hit"],

        "context_support_token_recall":
            row[
                "context_support_token_recall"
            ],

        "empty_context":
            row["empty_context"],

        "retrieved_doc_ids":
            " || ".join(

                item.get(
                    "doc_id",
                    ""
                )

                for item
                in row["retrieved"]
            ),

        "retrieved_titles":
            " || ".join(

                item.get(
                    "title",
                    ""
                )

                for item
                in row["retrieved"]
            ),
    }

    for row in retrieval_rows
])


retrieval_audit.to_csv(

    ART_DIR
    / "RAG_3_retrieval_audit.csv",

    index=False,

    encoding="utf-8-sig"
)


print(
    "Saved retrieval cache:",
    RETRIEVAL_CACHE_PATH
)

print(
    "Empty contexts:",
    int(
        retrieval_audit[
            "empty_context"
        ].sum()
    )
)


# ------------------------------------------------------------
# Free retrieval models before Qwen
# ------------------------------------------------------------

del embedder

if reranker_model is not None:
    del reranker_model

if reranker_tokenizer is not None:
    del reranker_tokenizer

gc.collect()
torch.cuda.empty_cache()


# ------------------------------------------------------------
# Load Qwen in 4-bit
# ------------------------------------------------------------

print(
    "\nLoading Qwen in 4-bit..."
)


quantization_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True
)


generator_tokenizer = (
    AutoTokenizer
    .from_pretrained(
        GENERATOR_NAME
    )
)


if generator_tokenizer.pad_token_id is None:

    generator_tokenizer.pad_token_id = (
        generator_tokenizer.eos_token_id
    )


generator_model = (
    AutoModelForCausalLM
    .from_pretrained(

        GENERATOR_NAME,

        quantization_config=
            quantization_config,

        device_map="auto",

        low_cpu_mem_usage=True
    )
)


generator_model.eval()


input_device = (

    generator_model

    .get_input_embeddings()

    .weight

    .device
)


# ------------------------------------------------------------
# Prompt and generation functions
# ------------------------------------------------------------

def build_context(retrieved):

    blocks = []


    for number, item in enumerate(
        retrieved,
        start=1
    ):

        blocks.append(

            f"[Context {number}]\n"

            f"শিরোনাম: "
            f"{item.get('title', '')}\n"

            f"বিষয়: "
            f"{item.get('topic', '')}\n"

            f"উৎস: "
            f"{item.get('source_url', '')}\n"

            f"তথ্য: "
            f"{item.get('text', '')}"
        )


    return "\n\n".join(
        blocks
    )


def generate_answer(
    question,
    retrieved
):

    context = build_context(
        retrieved
    )


    system_prompt = (

        "তুমি বাংলাদেশের সরকারি সেবা "
        "বিষয়ক প্রশ্নের উত্তরদাতা। "

        "শুধু দেওয়া Context-এর তথ্য "
        "ব্যবহার করবে। "

        "প্রশ্ন ও Context-এর ভাষা বা বাক্য "
        "এক না হলেও সমার্থক তথ্য শনাক্ত করবে। "

        "Context-এ উত্তর সমর্থনকারী তথ্য "
        "থাকলে অবশ্যই সরাসরি উত্তর দেবে; "

        "অযথা উত্তর পাওয়া যায়নি বলবে না। "

        "ফি, সময়, সংখ্যা ও প্রয়োজনীয় "
        "কাগজপত্র Context অনুযায়ী "
        "নির্ভুলভাবে লিখবে। "

        "উত্তর পরিষ্কার, সংক্ষিপ্ত এবং "
        "বাংলায় হবে। "

        "কেবল Context-এ প্রয়োজনীয় তথ্য "
        "একেবারেই না থাকলে বলবে: "

        "'প্রদত্ত তথ্যে এই প্রশ্নের "
        "উত্তর পাওয়া যায়নি।'"
    )


    user_prompt = (

        f"Context:\n{context}\n\n"

        f"Question:\n{question}\n\n"

        "বাংলায় সরাসরি উত্তর:"
    )


    messages = [

        {
            "role": "system",
            "content": system_prompt
        },

        {
            "role": "user",
            "content": user_prompt
        },
    ]


    prompt = (
        generator_tokenizer
        .apply_chat_template(

            messages,

            tokenize=False,

            add_generation_prompt=True
        )
    )


    inputs = generator_tokenizer(

        prompt,

        return_tensors="pt",

        truncation=True,

        max_length=7000
    )


    inputs = {

        key: value.to(
            input_device
        )

        for key, value
        in inputs.items()
    }


    with torch.inference_mode():

        output = generator_model.generate(

            **inputs,

            max_new_tokens=
                MAX_NEW_TOKENS,

            do_sample=False,

            repetition_penalty=1.05,

            pad_token_id=
                generator_tokenizer
                .pad_token_id,

            eos_token_id=
                generator_tokenizer
                .eos_token_id
        )


    generated_ids = output[

        0,

        inputs[
            "input_ids"
        ].shape[1]:

    ]


    answer = generator_tokenizer.decode(

        generated_ids,

        skip_special_tokens=True

    ).strip()


    ended_with_eos = bool(

        len(generated_ids) > 0

        and

        generated_ids[
            -1
        ].item()
        ==
        generator_tokenizer.eos_token_id
    )


    truncated = (

        len(generated_ids)
        >=
        MAX_NEW_TOKENS

        and

        not ended_with_eos
    )


    if not answer:

        answer = (
            "প্রদত্ত তথ্যে এই প্রশ্নের "
            "উত্তর পাওয়া যায়নি।"
        )


    return answer, truncated


# ------------------------------------------------------------
# Resume matching partial results
# ------------------------------------------------------------

done = {}


if PARTIAL_PATH.exists():

    with open(
        PARTIAL_PATH,
        "r",
        encoding="utf-8"
    ) as file:

        for line in file:

            line = line.strip()

            if not line:
                continue

            try:
                saved = json.loads(line)

            except json.JSONDecodeError:
                continue


            if (

                saved.get(
                    "corpus_sha256"
                )

                !=

                manifest[
                    "corpus_sha256"
                ]
            ):

                continue


            done[(
                saved["domain"],
                saved["id"]
            )] = saved


print(
    "Already completed from "
    "a matching partial run:",
    len(done)
)


# ------------------------------------------------------------
# Generate answers and save after every row
# ------------------------------------------------------------

for row in tqdm(
    retrieval_rows,
    desc="Qwen generation"
):

    key = (
        row["domain"],
        row["id"]
    )


    if key in done:
        continue


    answer, truncated = generate_answer(

        row["question"],

        row["retrieved"]
    )


    result = {

        "corpus_sha256":
            manifest[
                "corpus_sha256"
            ],

        "domain":
            row["domain"],

        "id":
            row["id"],

        "topic":
            row.get(
                "topic",
                ""
            ),

        "question_type":
            row.get(
                "question_type",
                ""
            ),

        "question":
            row["question"],

        "gold_answer":
            row["gold_answer"],

        "generated_answer":
            answer,

        "source_url":
            row.get(
                "source_url",
                ""
            ),

        "context_hit":
            bool(
                row[
                    "context_hit"
                ]
            ),

        "context_support_token_recall":
            float(
                row[
                    "context_support_token_recall"
                ]
            ),

        "empty_context":
            bool(
                row[
                    "empty_context"
                ]
            ),

        "truncated":
            bool(truncated),

        "retrieved_doc_ids":
            " || ".join(

                item.get(
                    "doc_id",
                    ""
                )

                for item
                in row["retrieved"]
            ),

        "retrieved_titles":
            " || ".join(

                item.get(
                    "title",
                    ""
                )

                for item
                in row["retrieved"]
            ),

        "retrieved_sources":
            " || ".join(

                item.get(
                    "source_url",
                    ""
                )

                for item
                in row["retrieved"]
            ),

        "retrieved_texts":
            " ||| ".join(

                item.get(
                    "text",
                    ""
                )

                for item
                in row["retrieved"]
            ),

        "rrf_scores":
            " || ".join(

                str(
                    item.get(
                        "rrf_score",
                        ""
                    )
                )

                for item
                in row["retrieved"]
            ),

        "reranker_scores":
            " || ".join(

                str(
                    item.get(
                        "reranker_score",
                        ""
                    )
                )

                for item
                in row["retrieved"]
            ),
    }


    with open(
        PARTIAL_PATH,
        "a",
        encoding="utf-8"
    ) as file:

        file.write(

            json.dumps(
                result,
                ensure_ascii=False
            )

            + "\n"
        )

        file.flush()

        os.fsync(
            file.fileno()
        )


    done[key] = result


# ------------------------------------------------------------
# Restore original test order and save final predictions
# ------------------------------------------------------------

ordered_results = []
missing_keys = []


for row in retrieval_rows:

    key = (
        row["domain"],
        row["id"]
    )

    if key not in done:

        missing_keys.append(key)

    else:

        ordered_results.append(
            done[key]
        )


if missing_keys:

    raise RuntimeError(
        f"Generation did not finish for "
        f"{len(missing_keys)} rows: "
        f"{missing_keys[:5]}"
    )


predictions = pd.DataFrame(
    ordered_results
)


predictions.to_csv(

    PREDICTIONS_CSV_PATH,

    index=False,

    encoding="utf-8-sig"
)


predictions.to_json(

    PREDICTIONS_JSON_PATH,

    orient="records",

    force_ascii=False,

    indent=2
)


no_answer_pattern = (
    r"উত্তর\s+"
    r"(?:পাওয়া|পাওয়া)\s+"
    r"(?:যায়নি|যায়নি)"
)


no_answer_count = int(

    predictions[
        "generated_answer"
    ]
    .astype(str)
    .str.contains(
        no_answer_pattern,
        regex=True
    )
    .sum()
)


print(
    "\nRAG-3 generation completed."
)

print(
    "Predictions:",
    len(predictions)
)

print(
    "No-answer responses:",
    no_answer_count
)

print(
    "No-answer percentage:",
    round(
        100
        *
        no_answer_count
        /
        len(predictions),
        2
    ),
    "%"
)

print(
    "Saved:",
    PREDICTIONS_CSV_PATH
)

print(
    "Saved:",
    PREDICTIONS_JSON_PATH
)


# Free Qwen before evaluation
del (
    generator_model,
    generator_tokenizer
)

gc.collect()
torch.cuda.empty_cache()

Loading BGE-M3 query encoder...
Loading multilingual BGE reranker...

Retrieving contexts for all test questions...


Hybrid retrieval:   0%|          | 0/248 [00:00<?, ?it/s]

Saved retrieval cache: /content/drive/MyDrive/govt_rag_artifacts/RAG_3/RAG_3_retrieval_cache.json
Empty contexts: 0

Loading Qwen in 4-bit...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Already completed from a matching partial run: 200


Qwen generation:   0%|          | 0/248 [00:00<?, ?it/s]


RAG-3 generation completed.
Predictions: 248
No-answer responses: 12
No-answer percentage: 4.84 %
Saved: /content/drive/MyDrive/govt_rag_artifacts/RAG_3/RAG_3_predictions.csv
Saved: /content/drive/MyDrive/govt_rag_artifacts/RAG_3/RAG_3_predictions.json


In [8]:
# ============================================================
# CELL 5 — SELF-CONTAINED RAG-3 EVALUATION
# Can be rerun later without Cells 1-4.
# ============================================================

%pip install -q \
    bert-score==0.3.13 \
    sacrebleu \
    rapidfuzz \
    nltk

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)

from collections import Counter
from pathlib import Path

import re
import unicodedata

import nltk
import numpy as np
import pandas as pd
import torch

from bert_score import score as bert_score
from IPython.display import display
from nltk.translate.meteor_score import meteor_score
from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

ART_DIR = Path(
    "/content/drive/MyDrive/"
    "govt_rag_artifacts/RAG_3"
)

PREDICTIONS_PATH = (
    ART_DIR
    / "RAG_3_predictions.csv"
)

RESULTS_PATH = (
    ART_DIR
    / "RAG_3_results.csv"
)


if not PREDICTIONS_PATH.exists():

    raise FileNotFoundError(
        f"Prediction file not found: "
        f"{PREDICTIONS_PATH}. "
        f"Run Cell 4 first."
    )


# ------------------------------------------------------------
# Load and validate predictions
# ------------------------------------------------------------

df = pd.read_csv(
    PREDICTIONS_PATH
).fillna("")


required_columns = {

    "domain",
    "id",
    "question",
    "gold_answer",
    "generated_answer",
}


missing_columns = (

    required_columns

    -

    set(df.columns)
)


if missing_columns:

    raise ValueError(
        f"Missing columns: "
        f"{sorted(missing_columns)}.\n"
        f"Available columns: "
        f"{df.columns.tolist()}"
    )


for column in required_columns:

    df[column] = (
        df[column]
        .astype(str)
    )


blank_gold = int(

    (
        df["gold_answer"]
        .str.strip()
        ==
        ""
    ).sum()
)


blank_predictions = int(

    (
        df["generated_answer"]
        .str.strip()
        ==
        ""
    ).sum()
)


print(
    "Loaded rows:",
    len(df)
)

print(
    "Blank gold answers:",
    blank_gold
)

print(
    "Blank generated answers:",
    blank_predictions
)


if blank_gold > 0:

    raise ValueError(
        "Evaluation stopped because "
        "gold answers are blank."
    )


# ------------------------------------------------------------
# Normalization
# ------------------------------------------------------------

BANGLA_DIGITS = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)


def normalize_text(text):

    value = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    value = value.translate(
        BANGLA_DIGITS
    ).lower()

    value = re.sub(
        r"[^\u0980-\u09FF"
        r"A-Za-z0-9]+",
        " ",
        value
    )

    return re.sub(
        r"\s+",
        " ",
        value
    ).strip()


def tokenize(text):

    value = normalize_text(text)

    return (
        value.split()
        if value
        else []
    )


# ------------------------------------------------------------
# Exact match and Token F1
# ------------------------------------------------------------

def normalized_exact_match(
    prediction,
    reference
):

    return float(

        normalize_text(prediction)

        ==

        normalize_text(reference)
    )


def token_f1(
    prediction,
    reference
):

    prediction_tokens = tokenize(
        prediction
    )

    reference_tokens = tokenize(
        reference
    )


    if (

        not prediction_tokens

        and

        not reference_tokens
    ):

        return 1.0


    if (

        not prediction_tokens

        or

        not reference_tokens
    ):

        return 0.0


    overlap = sum(

        (
            Counter(
                prediction_tokens
            )

            &

            Counter(
                reference_tokens
            )
        ).values()
    )


    if overlap == 0:
        return 0.0


    precision = (

        overlap

        /

        len(prediction_tokens)
    )


    recall = (

        overlap

        /

        len(reference_tokens)
    )


    return (

        2
        *
        precision
        *
        recall

        /

        (
            precision
            +
            recall
        )
    )


# ------------------------------------------------------------
# ROUGE
# ------------------------------------------------------------

def rouge_n_f1(
    prediction,
    reference,
    n
):

    prediction_tokens = tokenize(
        prediction
    )

    reference_tokens = tokenize(
        reference
    )


    if (

        len(prediction_tokens) < n

        or

        len(reference_tokens) < n
    ):

        return 0.0


    prediction_ngrams = Counter(

        tuple(
            prediction_tokens[
                index:index + n
            ]
        )

        for index in range(
            len(prediction_tokens)
            -
            n
            +
            1
        )
    )


    reference_ngrams = Counter(

        tuple(
            reference_tokens[
                index:index + n
            ]
        )

        for index in range(
            len(reference_tokens)
            -
            n
            +
            1
        )
    )


    overlap = sum(

        (
            prediction_ngrams

            &

            reference_ngrams
        ).values()
    )


    if overlap == 0:
        return 0.0


    precision = (

        overlap

        /

        sum(
            prediction_ngrams.values()
        )
    )


    recall = (

        overlap

        /

        sum(
            reference_ngrams.values()
        )
    )


    return (

        2
        *
        precision
        *
        recall

        /

        (
            precision
            +
            recall
        )
    )


def lcs_length(
    first,
    second
):

    previous = [
        0
    ] * (
        len(second)
        +
        1
    )


    for first_token in first:

        current = [0]


        for index, second_token in enumerate(
            second,
            start=1
        ):

            if first_token == second_token:

                current.append(
                    previous[
                        index - 1
                    ]
                    +
                    1
                )

            else:

                current.append(

                    max(

                        previous[index],

                        current[
                            index - 1
                        ]
                    )
                )


        previous = current


    return previous[-1]


def rouge_l_f1(
    prediction,
    reference
):

    prediction_tokens = tokenize(
        prediction
    )

    reference_tokens = tokenize(
        reference
    )


    if (

        not prediction_tokens

        or

        not reference_tokens
    ):

        return 0.0


    lcs = lcs_length(

        prediction_tokens,

        reference_tokens
    )


    precision = (

        lcs

        /

        len(prediction_tokens)
    )


    recall = (

        lcs

        /

        len(reference_tokens)
    )


    if precision + recall == 0:
        return 0.0


    return (

        2
        *
        precision
        *
        recall

        /

        (
            precision
            +
            recall
        )
    )


# ------------------------------------------------------------
# METEOR
# ------------------------------------------------------------

nltk.download(
    "wordnet",
    quiet=True
)

nltk.download(
    "omw-1.4",
    quiet=True
)


def meteor_value(
    prediction,
    reference
):

    prediction_tokens = tokenize(
        prediction
    )

    reference_tokens = tokenize(
        reference
    )


    if (

        not prediction_tokens

        or

        not reference_tokens
    ):

        return 0.0


    return float(

        meteor_score(

            [reference_tokens],

            prediction_tokens
        )
    )


def parse_bool(series):

    return (

        series
        .astype(str)
        .str.strip()
        .str.lower()
        .map({

            "true": True,
            "false": False,

            "1": True,
            "0": False,
        })
    )


# ------------------------------------------------------------
# Calculate row-level lexical metrics
# ------------------------------------------------------------

pairs = list(

    zip(

        df[
            "generated_answer"
        ],

        df[
            "gold_answer"
        ]
    )
)


df[
    "Normalized Exact Match"
] = [

    normalized_exact_match(
        prediction,
        reference
    )

    for prediction, reference
    in pairs
]


df[
    "Token F1"
] = [

    token_f1(
        prediction,
        reference
    )

    for prediction, reference
    in pairs
]


df[
    "Fuzzy Match"
] = [

    fuzz.token_set_ratio(

        normalize_text(
            prediction
        ),

        normalize_text(
            reference
        )

    ) / 100

    for prediction, reference
    in pairs
]


df[
    "ROUGE-1 F1"
] = [

    rouge_n_f1(
        prediction,
        reference,
        1
    )

    for prediction, reference
    in pairs
]


df[
    "ROUGE-2 F1"
] = [

    rouge_n_f1(
        prediction,
        reference,
        2
    )

    for prediction, reference
    in pairs
]


df[
    "ROUGE-L F1"
] = [

    rouge_l_f1(
        prediction,
        reference
    )

    for prediction, reference
    in pairs
]


df[
    "METEOR"
] = [

    meteor_value(
        prediction,
        reference
    )

    for prediction, reference
    in pairs
]


# ------------------------------------------------------------
# BERTScore
# CPU avoids GPU-memory conflict.
# ------------------------------------------------------------

print(
    "\nComputing BERTScore on CPU..."
)


bert_precision, bert_recall, bert_f1_values = bert_score(

    df[
        "generated_answer"
    ].astype(str).tolist(),

    df[
        "gold_answer"
    ].astype(str).tolist(),

    model_type=
        "bert-base-multilingual-cased",

    batch_size=4,

    device="cpu",

    verbose=True,

    idf=False,

    rescale_with_baseline=False
)


df[
    "BERT Precision"
] = (

    bert_precision
    .cpu()
    .numpy()
)


df[
    "BERT Recall"
] = (

    bert_recall
    .cpu()
    .numpy()
)


df[
    "BERT F1"
] = (

    bert_f1_values
    .cpu()
    .numpy()
)


# ------------------------------------------------------------
# Corpus BLEU
# ------------------------------------------------------------

bleu_metric = BLEU(

    tokenize="none",

    smooth_method="exp",

    effective_order=True
)


def calculate_corpus_bleu(
    group
):

    predictions = [

        " ".join(
            tokenize(text)
        )

        for text
        in group[
            "generated_answer"
        ]
    ]


    references = [

        " ".join(
            tokenize(text)
        )

        for text
        in group[
            "gold_answer"
        ]
    ]


    if (

        not any(predictions)

        or

        not any(references)
    ):

        return 0.0


    return (

        bleu_metric
        .corpus_score(

            predictions,

            [references]
        )
        .score

        /

        100
    )


# ------------------------------------------------------------
# Overall and per-domain results
# ------------------------------------------------------------

no_answer_pattern = (

    r"উত্তর\s+"

    r"(?:পাওয়া|পাওয়া)\s+"

    r"(?:যায়নি|যায়নি)"
)


def create_result_rows(
    group,
    scope,
    domain
):

    if "context_hit" in group.columns:

        context_hit = parse_bool(
            group["context_hit"]
        )

        valid_context = (
            context_hit.notna()
        )


        context_hits = int(

            (
                context_hit[
                    valid_context
                ]
                ==
                True
            ).sum()
        )


        context_misses = int(

            (
                context_hit[
                    valid_context
                ]
                ==
                False
            ).sum()
        )


        context_miss_rate = (

            context_misses

            /

            valid_context.sum()

            if valid_context.sum() > 0

            else np.nan
        )


    else:

        context_hits = 0
        context_misses = 0
        context_miss_rate = np.nan


    empty_contexts = (

        int(

            parse_bool(
                group[
                    "empty_context"
                ]
            )
            .fillna(False)
            .sum()
        )

        if "empty_context"
        in group.columns

        else 0
    )


    truncated_outputs = (

        int(

            parse_bool(
                group[
                    "truncated"
                ]
            )
            .fillna(False)
            .sum()
        )

        if "truncated"
        in group.columns

        else 0
    )


    no_answer_count = int(

        group[
            "generated_answer"
        ]
        .astype(str)
        .str.contains(

            no_answer_pattern,

            regex=True
        )
        .sum()
    )


    metrics = {

        "Normalized Exact Match":
            group[
                "Normalized Exact Match"
            ].mean(),

        "Token F1":
            group[
                "Token F1"
            ].mean(),

        "Fuzzy Match":
            group[
                "Fuzzy Match"
            ].mean(),

        "Corpus BLEU":
            calculate_corpus_bleu(
                group
            ),

        "ROUGE-1 F1":
            group[
                "ROUGE-1 F1"
            ].mean(),

        "ROUGE-2 F1":
            group[
                "ROUGE-2 F1"
            ].mean(),

        "ROUGE-L F1":
            group[
                "ROUGE-L F1"
            ].mean(),

        "METEOR":
            group[
                "METEOR"
            ].mean(),

        "BERT Precision":
            group[
                "BERT Precision"
            ].mean(),

        "BERT Recall":
            group[
                "BERT Recall"
            ].mean(),

        "BERT F1":
            group[
                "BERT F1"
            ].mean(),

        "Context Hit@5":
            context_hits,

        "Context Miss@5":
            context_misses,

        "Context Miss Rate":
            context_miss_rate,

        "Empty Contexts":
            empty_contexts,

        "Truncated Outputs":
            truncated_outputs,

        "No-answer Count":
            no_answer_count,

        "No-answer Rate":
            (
                no_answer_count
                /
                len(group)
            ),
    }


    return [

        {
            "scope":
                scope,

            "domain":
                domain,

            "count":
                len(group),

            "metric":
                metric,

            "score":
                float(score),
        }

        for metric, score
        in metrics.items()
    ]


result_rows = create_result_rows(

    df,

    "overall",

    "all"
)


for domain, domain_group in df.groupby(
    "domain",
    sort=True
):

    result_rows.extend(

        create_result_rows(

            domain_group,

            "domain",

            domain
        )
    )


results = pd.DataFrame(
    result_rows
)


# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

df.to_csv(

    PREDICTIONS_PATH,

    index=False,

    encoding="utf-8-sig"
)


results.to_csv(

    RESULTS_PATH,

    index=False,

    encoding="utf-8-sig"
)


print(
    "\nRAG-3 evaluation completed."
)

print(
    "Saved predictions with row metrics:",
    PREDICTIONS_PATH
)

print(
    "Saved result table:",
    RESULTS_PATH
)


display(

    results.loc[

        results["scope"]
        ==
        "overall",

        [
            "metric",
            "score"
        ]

    ].reset_index(
        drop=True
    )
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded rows: 248
Blank gold answers: 0
Blank generated answers: 0

Computing BERTScore on CPU...
calculating scores...
computing bert embedding.


  0%|          | 0/92 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/62 [00:00<?, ?it/s]

done in 36.99 seconds, 6.70 sentences/sec

RAG-3 evaluation completed.
Saved predictions with row metrics: /content/drive/MyDrive/govt_rag_artifacts/RAG_3/RAG_3_predictions.csv
Saved result table: /content/drive/MyDrive/govt_rag_artifacts/RAG_3/RAG_3_results.csv


,metric,score
0,Normalized Exact Match,0.205645
1,Token F1,0.673917
2,Fuzzy Match,0.877793
3,Corpus BLEU,0.487622
4,ROUGE-1 F1,0.673917
5,ROUGE-2 F1,0.572603
6,ROUGE-L F1,0.643415
7,METEOR,0.654045
8,BERT Precision,0.887929
9,BERT Recall,0.875262


In [6]:
# ============================================================
# REMOVE ONLY TRUNCATED ANSWERS FROM PARTIAL CHECKPOINT
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import pandas as pd
import json
import shutil
import os


ART_DIR = Path(
    "/content/drive/MyDrive/govt_rag_artifacts/RAG_3"
)

PREDICTIONS_PATH = ART_DIR / "RAG_3_predictions.csv"
PARTIAL_PATH = ART_DIR / "RAG_3_predictions_partial.jsonl"

if not PREDICTIONS_PATH.exists():
    raise FileNotFoundError(PREDICTIONS_PATH)

if not PARTIAL_PATH.exists():
    raise FileNotFoundError(PARTIAL_PATH)


df = pd.read_csv(PREDICTIONS_PATH).fillna("")

truncated_mask = (
    df["truncated"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["true", "1", "yes"])
)

truncated_rows = df.loc[
    truncated_mask,
    ["domain", "id"]
].copy()

regenerate_keys = set(
    zip(
        truncated_rows["domain"].astype(str),
        truncated_rows["id"].astype(str)
    )
)

print("Answers selected for regeneration:", len(regenerate_keys))


# Backup original partial checkpoint
backup_path = (
    ART_DIR /
    "RAG_3_predictions_partial_before_truncated_regeneration.jsonl"
)

shutil.copy2(
    PARTIAL_PATH,
    backup_path
)


# Remove only truncated rows from partial checkpoint
kept_rows = []
removed_rows = 0

with open(
    PARTIAL_PATH,
    "r",
    encoding="utf-8"
) as file:

    for line in file:

        line = line.strip()

        if not line:
            continue

        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            continue

        key = (
            str(row.get("domain", "")),
            str(row.get("id", ""))
        )

        if key in regenerate_keys:
            removed_rows += 1
        else:
            kept_rows.append(row)


temporary_path = PARTIAL_PATH.with_suffix(".tmp")

with open(
    temporary_path,
    "w",
    encoding="utf-8"
) as file:

    for row in kept_rows:
        file.write(
            json.dumps(
                row,
                ensure_ascii=False
            ) + "\n"
        )

os.replace(
    temporary_path,
    PARTIAL_PATH
)


print("Removed checkpoint rows:", removed_rows)
print("Kept completed answers:", len(kept_rows))
print("Backup saved:", backup_path)
print("\nNow change MAX_NEW_TOKENS to 320 and rerun Cell 4.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Answers selected for regeneration: 48
Removed checkpoint rows: 48
Kept completed answers: 200
Backup saved: /content/drive/MyDrive/govt_rag_artifacts/RAG_3/RAG_3_predictions_partial_before_truncated_regeneration.jsonl

Now change MAX_NEW_TOKENS to 320 and rerun Cell 4.
